In [0]:
print("Hello")

In [0]:
dbutils.widgets.text("start_date", "2015-01-01", "Start date (YYYY-MM-DD)")
dbutils.widgets.text("end_date",   "2015-01-09", "End date (YYYY-MM-DD)")
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental", "backfill"], "Load type")

In [0]:
import gzip, zipfile, io

path = "/Volumes/workspace/bronze/raw_landing/mkt_summary/2015/2015-01-02.Z"

data = open(path, "rb").read()
print("First bytes:", data[:2], "| size:", len(data), "bytes")

if data[:2] == b"PK":
    z = zipfile.ZipFile(io.BytesIO(data))
    print("ZIP containing:", z.namelist())
    text = z.read(z.namelist()[0]).decode("latin-1")
else:
    text = gzip.decompress(data).decode("latin-1")

lines = text.splitlines()
print(len(lines), "rows")
for line in lines[:5]:
    print(line)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
mkt_summary_schema = StructType([
    StructField('trade_date',StringType(),True),
    StructField('symbol',StringType(),True),
    StructField('sector_code',StringType(),True),
    StructField('company_name',StringType(),True),
    StructField('open_raw',StringType(),True),
    StructField('high_raw',StringType(),True),
    StructField('low_raw',StringType(),True),
    StructField('close_raw',StringType(),True),
    StructField("volume_raw",StringType(), True),
    StructField("ldcp_raw",StringType(), True),  
    StructField("extra_1",StringType(), True),   
    StructField("extra_2",StringType(), True),   
    StructField("extra_3",StringType(), True)
])

In [0]:
import gzip, zipfile, io, os

src = "/Volumes/workspace/bronze/raw_landing/mkt_summary/2015/2015-01-02.Z"
out = "/Volumes/workspace/bronze/raw_landing/_extracted/2015/2015-01-02.txt"

data = open(src, "rb").read()
if data[:2] == b"PK":
    z = zipfile.ZipFile(io.BytesIO(data))
    text = z.read(z.namelist()[0]).decode("latin-1")
else:
    text = gzip.decompress(data).decode("latin-1")

os.makedirs(os.path.dirname(out), exist_ok=True)
with open(out, "w", encoding="utf-8") as f:
    f.write(text)

print("Wrote", len(text.splitlines()), "lines to", out)

In [0]:
df = (spark.read
      .schema(mkt_summary_schema)
      .option("sep", "|")
      .option("header", "false")
      .csv(out))

df.printSchema()
print("Rows:", df.count())
display(df.limit(10))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.bronze.mkt_summary (
  trade_date       STRING,
  symbol           STRING,
  sector_code      STRING,
  company_name     STRING,
  open_raw         STRING,
  high_raw         STRING,
  low_raw          STRING,
  close_raw        STRING,
  volume_raw       STRING,
  ldcp_raw         STRING,
  extra_1          STRING,
  extra_2          STRING,
  extra_3          STRING,
  file_trade_date  DATE,
  source_file      STRING,
  batch_id         STRING,
  load_timestamp   TIMESTAMP
)
USING DELTA
PARTITIONED BY (file_trade_date)

In [0]:
from pyspark.sql import functions as F


df_bronze = (df
    .withColumn("file_trade_date", F.lit("2015-01-02").cast("date"))
    .withColumn("source_file",     F.col("_metadata.file_path"))
    .withColumn("batch_id",        F.lit("test_run_1"))
    .withColumn("load_timestamp",  F.current_timestamp())
)

df_bronze = df_bronze.select(
    "trade_date", "symbol", "sector_code", "company_name",
    "open_raw", "high_raw", "low_raw", "close_raw", "volume_raw", "ldcp_raw",
    "extra_1", "extra_2", "extra_3",
    "file_trade_date", "source_file", "batch_id", "load_timestamp"
)

display(df_bronze.limit(5))

In [0]:
# Step 2: write to the Bronze table
df_bronze.write.mode("append").saveAsTable("workspace.bronze.mkt_summary")
print("Written:", df_bronze.count(), "rows")

In [0]:
%sql

SELECT COUNT(*) AS rows, MIN(load_timestamp), MAX(load_timestamp)
FROM workspace.bronze.mkt_summary;

SELECT * FROM workspace.bronze.mkt_summary LIMIT 10;

In [0]:
import gzip, zipfile, io, os
from pyspark.sql import functions as F

def load_to_bronze(date, batch_id, load_type):
  print("Loading", date, "with batch_id", batch_id, "and load_type", load_type)
  year = date[:4]
  src = "/Volumes/workspace/bronze/raw_landing/mkt_summary/" + year + "/" + date + ".Z"
  out = "/Volumes/workspace/bronze/raw_landing/_extracted/" + year + "/" + date + ".txt"

  # 1. Decompress .Z (ZIP or GZIP) to a staging .txt file
  data = open(src, "rb").read()
  if data[:2] == b"PK":
      z = zipfile.ZipFile(io.BytesIO(data))
      text = z.read(z.namelist()[0]).decode("latin-1")
  else:
      text = gzip.decompress(data).decode("latin-1")

  os.makedirs(os.path.dirname(out), exist_ok=True)
  with open(out, "w", encoding="utf-8") as f:
      f.write(text)

  print("Wrote", len(text.splitlines()), "lines to", out)

  # 2. Read with the explicit schema
  df = (spark.read
        .schema(mkt_summary_schema)
        .option("sep", "|")
        .option("header", "false")
        .csv(out))

  # 3. Add label columns
  df_bronze = (df
    .withColumn("file_trade_date", F.lit(date).cast("date"))
    .withColumn("source_file",     F.lit(src))
    .withColumn("batch_id",        F.lit(batch_id))
    .withColumn("load_timestamp",  F.current_timestamp())
  )

  df_bronze = df_bronze.select(
    "trade_date", "symbol", "sector_code", "company_name",
    "open_raw", "high_raw", "low_raw", "close_raw", "volume_raw", "ldcp_raw",
    "extra_1", "extra_2", "extra_3",
    "file_trade_date", "source_file", "batch_id", "load_timestamp"
  )

  # 4. Write to Bronze
  (df_bronze.write
     .mode("overwrite")
     .option("replaceWhere", f"file_trade_date = '{date}'")
     .saveAsTable("workspace.bronze.mkt_summary"))

  # 5. Return number of rows written
  row_count = df_bronze.count()
  print("Loaded", row_count, "rows for", date)
  return row_count

In [0]:
rows = load_to_bronze("2015-01-05", "test", "backfill")

In [0]:
%sql
SELECT file_trade_date, batch_id, COUNT(*) AS rows
FROM workspace.bronze.mkt_summary
GROUP BY file_trade_date, batch_id
ORDER BY file_trade_date

In [0]:
   load_to_bronze("2015-01-05", "test2", "backfill")

In [0]:
import os
import datetime as dt

def load_range(start_date, end_date, load_type):

    batch_id = f"bronze_{dt.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
    day  = dt.date.fromisoformat(start_date)
    last = dt.date.fromisoformat(end_date)

    total_rows = 0
    days_loaded = 0
    days_missing = 0
    days_failed = 0                                         # NEW

    while day <= last:
        d = day.isoformat()                 # e.g. "2015-01-02"

        if day.weekday() < 5:               # Mon-Fri only (5 = Sat, 6 = Sun)
            year = d[:4]
            src = "/Volumes/workspace/bronze/raw_landing/mkt_summary/" + year + "/" + d + ".Z"

            if os.path.exists(src):
                start = dt.datetime.now()                   # NEW
                try:                                        # NEW
                    rows = load_to_bronze(d, batch_id, load_type)
                    write_log(batch_id, "raw_to_bronze", load_type, src,
                              start, dt.datetime.now(), "SUCCESS",
                              rows_inserted=rows)
                    total_rows += rows
                    days_loaded += 1
                except Exception as e:                      # NEW
                    write_log(batch_id, "raw_to_bronze", load_type, src,
                              start, dt.datetime.now(), "FAILURE",
                              error_message=str(e)[:1000])
                    print("FAILED:", d, "-", str(e)[:200])
                    days_failed += 1
            else:
                print("No file (holiday or not uploaded):", d)
                days_missing += 1

        day = day + dt.timedelta(days=1)

    print(f"Done. batch={batch_id}, days loaded={days_loaded}, "
          f"failed={days_failed}, weekdays without file={days_missing}, "
          f"total rows={total_rows}")
    return batch_id

In [0]:
load_range("2015-01-01", "2015-01-09", "backfill")

In [0]:
%sql
SELECT file_trade_date, batch_id, COUNT(*) AS rows
FROM workspace.bronze.mkt_summary
GROUP BY file_trade_date, batch_id
ORDER BY file_trade_date

In [0]:
start_date = dbutils.widgets.get("start_date")
end_date   = dbutils.widgets.get("end_date")
load_type  = dbutils.widgets.get("load_type")

load_range(start_date, end_date, load_type)

In [0]:
%sql
DROP TABLE IF EXISTS workspace.ops.pipeline_execution_logs;

CREATE TABLE workspace.ops.pipeline_execution_logs (
    batch_id          STRING,
    layer             STRING,
    load_type         STRING,
    parameter         STRING,
    start_time        TIMESTAMP,
    end_time          TIMESTAMP,
    duration_seconds  DOUBLE,
    status            STRING,
    rows_inserted     BIGINT,
    rows_updated      BIGINT,
    rows_quarantined  BIGINT,
    error_message     STRING,
    load_timestamp    TIMESTAMP
);

In [0]:
import datetime as dt
from pyspark.sql.types import (StructType, StructField, StringType,
                               TimestampType, DoubleType, LongType)

log_schema = StructType([
    StructField("batch_id",         StringType(),    True),
    StructField("layer",            StringType(),    True),
    StructField("load_type",        StringType(),    True),
    StructField("parameter",        StringType(),    True),
    StructField("start_time",       TimestampType(), True),
    StructField("end_time",         TimestampType(), True),
    StructField("duration_seconds", DoubleType(),    True),
    StructField("status",           StringType(),    True),
    StructField("rows_inserted",    LongType(),      True),
    StructField("rows_updated",     LongType(),      True),
    StructField("rows_quarantined", LongType(),      True),
    StructField("error_message",    StringType(),    True),
    StructField("load_timestamp",   TimestampType(), True),
])


In [0]:

def write_log(batch_id, layer, load_type, parameter, start_time, end_time,
              status, rows_inserted=0, rows_updated=0, rows_quarantined=0,
              error_message=None):
    duration = (end_time - start_time).total_seconds()
    row = (batch_id, layer, load_type, parameter, start_time, end_time,
           duration, status, int(rows_inserted), int(rows_updated),
           int(rows_quarantined), error_message, dt.datetime.now())

    (spark.createDataFrame([row], log_schema)
          .write.mode("append")
          .saveAsTable("workspace.ops.pipeline_execution_logs"))

In [0]:
t0 = dt.datetime.now()
t1 = dt.datetime.now()
write_log("test_batch", "raw_to_bronze", "backfill", "test", t0, t1, "SUCCESS", rows_inserted=10)

In [0]:
%sql
SELECT * FROM workspace.ops.pipeline_execution_logs

In [0]:
%sql
DELETE FROM workspace.ops.pipeline_execution_logs WHERE batch_id = 'test_batch'

In [0]:
%sql
SELECT batch_id, parameter, status, rows_inserted, duration_seconds, error_message
FROM workspace.ops.pipeline_execution_logs
ORDER BY start_time DESC